# 7.2 交叉验证

> **模块七：模型评估与优化** | NOAI竞赛课程
>
> 本节约 **2课时**，重点掌握交叉验证方法与超参数搜索

---

## 📚 学习目标

1. 理解为什么单次划分的评估结果不可靠
2. 掌握K折交叉验证（K-Fold Cross Validation）的原理
3. 了解分层K折交叉验证（Stratified K-Fold）的必要性
4. 理解留一法（LOOCV）及其适用场景
5. 掌握网格搜索（Grid Search）和随机搜索（Random Search）
6. 使用sklearn进行交叉验证实战和超参数调优

---

## 1. 为什么需要交叉验证？

### 1.1 单次划分的局限性

在之前的课程中，我们通常将数据**一次划分**为训练集和测试集：

```
全部数据 -> [训练集 70%] + [测试集 30%]
```

这种方法存在以下问题：

| 问题 | 说明 |
|------|------|
| **评估结果不稳定** | 不同的随机种子会导致不同的划分，模型表现波动大 |
| **数据利用不充分** | 测试集数据不参与训练，在小数据集上浪费严重 |
| **对划分敏感** | 恰好某类全在测试集，评估结果异常 |

### 1.2 交叉验证的思想

**核心思想**：将数据多次划分，每次用不同的子集作为验证集，最终取平均。

这样每个样本都有机会被用作训练和验证，评估结果更稳定、更可靠。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 生成数据集（较小，便于展示问题）
X, y = make_classification(n_samples=200, n_features=10, n_informative=5,
                          random_state=42)

# 演示单次划分的不稳定性
print("=== 单次划分的不稳定性演示 ===")
print(f"{'随机种子':>8} | {'训练集准确率':>14} | {'测试集准确率':>14}")
print("-" * 46)

test_accs = []
for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    test_accs.append(test_acc)
    if seed < 10:
        print(f"  {seed:>6d} | {train_acc:>14.4f} | {test_acc:>14.4f}")

print(f"\n20次划分的测试准确率：均值={np.mean(test_accs):.4f}，标准差={np.std(test_accs):.4f}")
print(f"波动范围：{min(test_accs):.4f} ~ {max(test_accs):.4f}")
print("\n⚠️ 仅一次划分的结果可能不准确！")

---

## 2. K折交叉验证（K-Fold Cross Validation）

### 2.1 原理

将数据集分成 **K个大小相等（或近似相等）的子集（fold）**：

$$\text{数据集} = \text{Fold}_1 \cup \text{Fold}_2 \cup \cdots \cup \text{Fold}_K$$

进行 **K轮** 训练和验证：
- 第1轮：用Fold 2~K训练，Fold 1验证
- 第2轮：用Fold 1,3~K训练，Fold 2验证
- ...
- 第K轮：用Fold 1~K-1训练，Fold K验证

最终评估指标为K轮结果的**平均值**：

$$\text{CV Score} = \frac{1}{K}\sum_{i=1}^{K} \text{Score}_i$$

### 2.2 常见K值选择

| K值 | 名称 | 特点 |
|-----|------|------|
| K=5 | 5折交叉验证 | 最常用，平衡偏差与方差 |
| K=10 | 10折交叉验证 | 偏差更小，计算量更大 |
| K=N | 留一法 | 每次留1个样本验证 |

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

# 演示K折交叉验证的划分过程
print("=== 5折交叉验证划分示意 ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"第{fold}折: 训练集 {len(train_idx)} 个样本, 验证集 {len(val_idx)} 个样本")

print(f"\n验证集索引示例：")
for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"  第{fold}折验证集前10个索引: {val_idx[:10].tolist()}")

# 可视化K折划分
n_splits = 5
kf_vis = KFold(n_splits=n_splits, shuffle=False)

fig, ax = plt.subplots(figsize=(14, 4))
for i, (train_idx, val_idx) in enumerate(kf_vis.split(X)):
    # 训练集用蓝色
    for idx in train_idx:
        ax.barh(i, 1, left=idx, color='#3498DB', alpha=0.6)
    # 验证集用红色
    for idx in val_idx:
        ax.barh(i, 1, left=idx, color='#E74C3C', alpha=0.8)

ax.set_xlabel('样本索引', fontsize=12)
ax.set_ylabel('折数', fontsize=12)
ax.set_yticks(range(n_splits))
ax.set_yticklabels([f'第{i+1}折' for i in range(n_splits)])
ax.set_title('K折交叉验证划分示意（蓝色=训练集，红色=验证集）', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 使用cross_val_score进行交叉验证
model = LogisticRegression(max_iter=1000)

# 5折交叉验证
cv_scores_5 = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("=== 5折交叉验证结果 ===")
for i, score in enumerate(cv_scores_5, 1):
    print(f"  第{i}折准确率: {score:.4f}")

print(f"\n平均准确率: {cv_scores_5.mean():.4f} +/- {cv_scores_5.std():.4f}")

# 对比不同K值
print("\n=== 不同K值的交叉验证对比 ===")
print(f"{'K值':>6} | {'平均准确率':>12} | {'标准差':>10}")
print("-" * 38)

for k in [3, 5, 10, 15, 20]:
    scores = cross_val_score(model, X, y, cv=k, scoring='accuracy')
    print(f"  {k:>4d} | {scores.mean():>12.4f} | {scores.std():>10.4f}")

print("\n📌 K越大 -> 单次训练数据越多（偏差下降），但折数多（方差可能上升）")
print("📌 常用K=5或K=10是实践中的良好折中")

---

## 3. 分层K折交叉验证（Stratified K-Fold）

### 3.1 问题：普通K折在类别不平衡时的缺陷

当数据类别不平衡时，普通K折可能产生某些折中某类样本过多或过少的问题。

例如：200个样本中正类30个、负类170个
- 普通K折5折：某折可能只有2个正类样本，甚至0个！
- 分层K折：**保证每折中正类/负类比例相同**

### 3.2 Stratified K-Fold 原理

在每个fold中**保持各类别的比例与原始数据集一致**：

$$\frac{n_k^{(i)}}{n^{(i)}} \approx \frac{N_k}{N} \quad \forall k, i$$

其中 $n_k^{(i)}$ 是第 $i$ 折中第 $k$ 类的样本数，$N_k$ 是数据集中第 $k$ 类的总数。

> 🎯 **NOAI竞赛建议**：分类任务中，**始终优先使用StratifiedKFold**！

In [ ]:
from sklearn.model_selection import StratifiedKFold

# 创建不平衡数据
X_imb, y_imb = make_classification(
    n_samples=200, n_features=10, weights=[0.85, 0.15], random_state=42)

print(f"数据集类别分布: 正类={sum(y_imb==1)}, 负类={sum(y_imb==0)}")
print(f"正类比例: {sum(y_imb==1)/len(y_imb)*100:.1f}%")

# 对比普通K折 vs 分层K折
kf_normal = KFold(n_splits=5, shuffle=True, random_state=42)
kf_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n=== 普通 K-Fold 各折类别分布 ===")
for fold, (_, val_idx) in enumerate(kf_normal.split(X_imb), 1):
    y_val = y_imb[val_idx]
    pos_ratio = sum(y_val==1)/len(y_val)*100
    print(f"  第{fold}折: 总数{len(y_val)}, 正类{sum(y_val==1)}, 正类比例{pos_ratio:.1f}%")

print("\n=== 分层 Stratified K-Fold 各折类别分布 ===")
for fold, (_, val_idx) in enumerate(kf_stratified.split(X_imb, y_imb), 1):
    y_val = y_imb[val_idx]
    pos_ratio = sum(y_val==1)/len(y_val)*100
    print(f"  第{fold}折: 总数{len(y_val)}, 正类{sum(y_val==1)}, 正类比例{pos_ratio:.1f}%")

# 对比两种方法的CV分数稳定性
print("\n=== 两种CV方法的分数稳定性对比 ===")

scores_normal = cross_val_score(model, X_imb, y_imb, cv=kf_normal, scoring='f1')
scores_stratified = cross_val_score(model, X_imb, y_imb, cv=kf_stratified, scoring='f1')

print(f"普通K-Fold:    F1 = {scores_normal.mean():.4f} +/- {scores_normal.std():.4f}")
print(f"分层K-Fold:    F1 = {scores_stratified.mean():.4f} +/- {scores_stratified.std():.4f}")

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 6), scores_normal, color='#E74C3C', alpha=0.7)
axes[0].axhline(y=scores_normal.mean(), color='#E74C3C', linestyle='--', linewidth=2)
axes[0].set_title(f'普通K-Fold F1分数\n均值={scores_normal.mean():.4f}, 标准差={scores_normal.std():.4f}', fontsize=12)
axes[0].set_xlabel('折数'); axes[0].set_ylabel('F1分数')
axes[0].set_ylim(0, 1)

axes[1].bar(range(1, 6), scores_stratified, color='#27AE60', alpha=0.7)
axes[1].axhline(y=scores_stratified.mean(), color='#27AE60', linestyle='--', linewidth=2)
axes[1].set_title(f'分层K-Fold F1分数\n均值={scores_stratified.mean():.4f}, 标准差={scores_stratified.std():.4f}', fontsize=12)
axes[1].set_xlabel('折数'); axes[1].set_ylabel('F1分数')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

---

## 4. 留一法（Leave-One-Out Cross Validation, LOOCV）

### 4.1 原理

留一法是K折交叉验证的极端情况，即 **K = N**（N为样本总数）：
- 每次只留1个样本作为验证集
- 其余N-1个样本全部用于训练
- 重复N次，每个样本恰好被验证一次

$$\text{LOOCV Score} = \frac{1}{N}\sum_{i=1}^{N} L(y_i, \hat{f}^{(-i)}(x_i))$$

### 4.2 优缺点

| 优点 | 缺点 |
|------|------|
| 训练数据利用最大化（N-1个样本） | 计算量巨大（需训练N个模型） |
| 评估结果几乎无偏 | 评估方差较大（各验证集高度相关） |
| 适合小数据集 | N>1000时不实用 |

In [ ]:
from sklearn.model_selection import LeaveOneOut
import time

# 使用小数据集演示LOOCV
X_small, y_small = make_classification(n_samples=50, n_features=5,
                                        n_informative=3, random_state=42)

loo = LeaveOneOut()
print(f"=== 留一法（LOOCV）===")
print(f"样本总数: {len(X_small)}")
print(f"需要训练 {loo.get_n_splits(X_small)} 个模型！")

# 计时
start = time.time()
loo_scores = cross_val_score(model, X_small, y_small, cv=loo, scoring='accuracy')
elapsed = time.time() - start

print(f"\nLOOCV准确率: {loo_scores.mean():.4f} +/- {loo_scores.std():.4f}")
print(f"耗时: {elapsed:.2f}秒")

# 对比5折CV
start5 = time.time()
cv5_scores = cross_val_score(model, X_small, y_small, cv=5, scoring='accuracy')
elapsed5 = time.time() - start5

print(f"\n5折CV准确率:   {cv5_scores.mean():.4f} +/- {cv5_scores.std():.4f}")
print(f"耗时: {elapsed5:.2f}秒")
print(f"\nLOOCV是5折CV耗时的大约 {elapsed/elapsed5:.1f} 倍")

---

## 5. 超参数搜索

### 5.1 什么是超参数？

| 类型 | 定义 | 举例 |
|------|------|------|
| **模型参数** | 训练过程中学到的参数 | 线性回归的权重 $w$、偏置 $b$ |
| **超参数** | 训练前需要人为设定的参数 | 正则化系数 $\lambda$、树的深度、K近邻的K |

超参数的选择直接影响模型性能，但我们无法通过训练自动确定最优值，需要通过**搜索**来寻找。

### 5.2 网格搜索（Grid Search）

**穷举搜索**超参数空间中的所有组合：

```
参数网格: C = [0.01, 0.1, 1, 10]
          penalty = ['l1', 'l2']

总组合数: 4 x 2 = 8 种
每种组合做5折交叉验证 -> 共 8 x 5 = 40 次训练
```

**优点**：保证找到搜索范围内的最优组合
**缺点**：组合爆炸，当参数多时计算量巨大

### 5.3 随机搜索（Random Search）

在超参数空间中**随机采样**指定数量的组合：

**优点**：
- 相同计算预算下，通常比网格搜索覆盖更多区域
- 研究表明：对大多数问题，随机搜索能以更少的迭代找到接近最优的解

**缺点**：
- 不保证找到搜索范围内的绝对最优
- 可能错过某些重要区域

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC
from scipy.stats import loguniform
import time

# 使用标准化数据
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=500, n_features=20, n_informative=10,
                          random_state=42)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============ 网格搜索 ============
print("=" * 60)
print("            网格搜索 (Grid Search)")
print("=" * 60)

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['rbf']
}

print(f"参数网格:")
for key, values in param_grid.items():
    print(f"  {key}: {values}")
total_combos = np.prod([len(v) for v in param_grid.values()])
print(f"\n总组合数: {total_combos}")
print(f"5折CV总训练次数: {total_combos * 5}")

grid_search = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy',
                          return_train_score=True, n_jobs=-1)

start_grid = time.time()
grid_search.fit(X_scaled, y)
grid_time = time.time() - start_grid

print(f"\n⏱  耗时: {grid_time:.2f}秒")
print(f"🏆 最佳参数: {grid_search.best_params_}")
print(f"🏆 最佳分数: {grid_search.best_score_:.4f}")

In [ ]:
# ============ 随机搜索 ============
print("\n" + "=" * 60)
print("           随机搜索 (Random Search)")
print("=" * 60)

param_dist = {
    'C': loguniform(1e-2, 1e2),
    'gamma': loguniform(1e-3, 1e0),
    'kernel': ['rbf']
}

n_iter = 20  # 只尝试20种组合
print(f"搜索空间: C ~ LogUniform(0.01, 100)")
print(f"          gamma ~ LogUniform(0.001, 1)")
print(f"采样次数: {n_iter}")
print(f"5折CV总训练次数: {n_iter * 5}")

random_search = RandomizedSearchCV(
    SVC(), param_dist, n_iter=n_iter, cv=5, scoring='accuracy',
    return_train_score=True, random_state=42, n_jobs=-1)

start_rand = time.time()
random_search.fit(X_scaled, y)
rand_time = time.time() - start_rand

print(f"\n⏱  耗时: {rand_time:.2f}秒")
print(f"🏆 最佳参数: {random_search.best_params_}")
print(f"🏆 最佳分数: {random_search.best_score_:.4f}")

In [ ]:
# ============ 结果可视化对比 ============
print("\n" + "=" * 60)
print("              方法对比")
print("=" * 60)

print(f"{'方法':<15} {'最佳分数':>10} {'耗时(秒)':>10}")
print("-" * 40)
print(f"{'网格搜索':<15} {grid_search.best_score_:>10.4f} {grid_time:>10.2f}")
print(f"{'随机搜索':<15} {random_search.best_score_:>10.4f} {rand_time:>10.2f}")

# 可视化网格搜索的热力图
import pandas as pd

results = pd.DataFrame(grid_search.cv_results_)

# 提取C和gamma的交叉验证结果
pivot_table = results.pivot_table(
    values='mean_test_score',
    index='param_gamma',
    columns='param_C'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 热力图
import seaborn as sns
sns.heatmap(pivot_table.astype(float), annot=True, fmt='.4f', cmap='YlOrRd',
            ax=axes[0], linewidths=0.5)
axes[0].set_title('网格搜索: C vs gamma 的准确率热力图', fontsize=13)
axes[0].set_xlabel('C (正则化系数)')
axes[0].set_ylabel('gamma')

# 随机搜索采样点
rand_results = pd.DataFrame(random_search.cv_results_)
axes[1].scatter(rand_results['param_C'], rand_results['param_gamma'],
                c=rand_results['mean_test_score'], cmap='YlOrRd', s=150, edgecolors='black')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('C (log scale)')
axes[1].set_ylabel('gamma (log scale)')
axes[1].set_title('随机搜索: 采样点分布（颜色=分数）', fontsize=13)

plt.tight_layout()
plt.show()

---

## 6. 完整实战：Pipeline + 交叉验证 + 网格搜索

在实际项目中，我们通常将**数据预处理、特征工程和模型训练**组合成一个Pipeline，再与交叉验证和超参数搜索结合使用。

> 🎯 这是NOAI竞赛中推荐的建模流程！

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

# ====== 构建完整Pipeline ======
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 定义参数网格（注意使用 '步骤名__参数名' 格式）
full_param_grid = {
    'pca__n_components': [5, 10, 15, 20],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 10, None],
    'classifier__min_samples_split': [2, 5, 10]
}

# 使用更大数据集
X_full, y_full = make_classification(
    n_samples=1000, n_features=30, n_informative=15,
    n_redundant=5, random_state=42)

total_combos = np.prod([len(v) for v in full_param_grid.values()])
print(f"参数组合总数: {total_combos}")
print(f"5折CV总训练次数: {total_combos * 5}")
print("\n⏳ 执行网格搜索...")

# 执行搜索
full_grid = GridSearchCV(
    pipeline, full_param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

full_grid.fit(X_full, y_full)

print(f"\n🏆 最佳参数:")
for param, value in full_grid.best_params_.items():
    print(f"  {param}: {value}")
print(f"\n🏆 最佳CV分数: {full_grid.best_score_:.4f}")

# 使用cross_validate获取更详细的评估
print("\n=== 最优模型的详细交叉验证结果 ===")
best_model = full_grid.best_estimator_
cv_results = cross_validate(best_model, X_full, y_full,
                            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                            scoring=['accuracy', 'precision', 'recall', 'f1'],
                            return_train_score=True)

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    train_scores = cv_results[f'train_{metric}']
    test_scores = cv_results[f'test_{metric}']
    print(f"{metric:>10}: 训练={train_scores.mean():.4f}+/-{train_scores.std():.4f}, "
          f"验证={test_scores.mean():.4f}+/-{test_scores.std():.4f}")

---

## 7. 练习题

### 📝 练习1：交叉验证稳定性对比

选择一个sklearn内置数据集，对比以下情况下的评估结果：
- 单次划分（不同随机种子重复10次）
- 5折交叉验证
- 10折交叉验证

记录每种方法的均值和标准差。

In [ ]:
# 练习1：稳定性对比
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
X_c, y_c = data.data, data.target

# 单次划分（10次不同种子）
single_scores = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.3, random_state=seed)
    dt = DecisionTreeClassifier(random_state=seed)
    dt.fit(X_tr, y_tr)
    single_scores.append(accuracy_score(y_te, dt.predict(X_te)))

# 5折和10折交叉验证
cv5_scores = cross_val_score(DecisionTreeClassifier(), X_c, y_c, cv=5, scoring='accuracy')
cv10_scores = cross_val_score(DecisionTreeClassifier(), X_c, y_c, cv=10, scoring='accuracy')

print(f"{'方法':<20} {'均值':>8} {'标准差':>8} {'最小值':>8} {'最大值':>8}")
print("-" * 50)
print(f"{'单次划分(10次)':<20} {np.mean(single_scores):>8.4f} {np.std(single_scores):>8.4f} "
      f"{np.min(single_scores):>8.4f} {np.max(single_scores):>8.4f}")
print(f"{'5折CV':<20} {cv5_scores.mean():>8.4f} {cv5_scores.std():>8.4f} "
      f"{cv5_scores.min():>8.4f} {cv5_scores.max():>8.4f}")
print(f"{'10折CV':<20} {cv10_scores.mean():>8.4f} {cv10_scores.std():>8.4f} "
      f"{cv10_scores.min():>8.4f} {cv10_scores.max():>8.4f}")

print("\n📌 结论: 交叉验证的标准差通常更小，评估更稳定")

### 📝 练习2：超参数搜索实战

对随机森林分类器进行超参数搜索，找到最佳参数组合。

要求：
1. 使用 `make_classification` 生成数据
2. 分别使用 GridSearchCV 和 RandomizedSearchCV
3. 搜索 `n_estimators`, `max_depth`, `min_samples_split`
4. 对比两种方法的结果和耗时

In [ ]:
# 练习2：超参数搜索实战
X_ex, y_ex = make_classification(n_samples=800, n_features=20,
                                n_informative=12, random_state=42)

rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

# Grid Search
print("⏳ Grid Search...")
t0 = time.time()
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42),
                      rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_ex, y_ex)
t_grid = time.time() - t0

# Random Search
from scipy.stats import randint
rf_param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [3, 5, 10, 15, 20, None],
    'min_samples_split': randint(2, 20)
}

print("⏳ Random Search...")
t0 = time.time()
rand_rf = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                             rf_param_dist, n_iter=30, cv=5,
                             scoring='accuracy', random_state=42, n_jobs=-1)
rand_rf.fit(X_ex, y_ex)
t_rand = time.time() - t0

print(f"\n{'方法':<15} {'最佳分数':>10} {'耗时(秒)':>10} {'最佳参数'}")
print("-" * 70)
print(f"{'Grid Search':<15} {grid_rf.best_score_:>10.4f} {t_grid:>10.2f} {grid_rf.best_params_}")
print(f"{'Random Search':<15} {rand_rf.best_score_:>10.4f} {t_rand:>10.2f} {rand_rf.best_params_}")

### 📝 练习3（挑战）：自定义CV策略

在某些竞赛中，数据具有时间相关性（如NOAI中可能的时序任务）。此时需要使用 **时间序列交叉验证（TimeSeriesSplit）**。

请尝试使用 `sklearn.model_selection.TimeSeriesSplit`，并对比它与普通K折的区别。

In [ ]:
# 练习3：时间序列交叉验证
from sklearn.model_selection import TimeSeriesSplit

# 生成模拟时序数据
np.random.seed(42)
n_samples = 200
X_ts = np.random.randn(n_samples, 5)
# 添加趋势
y_ts = (X_ts[:, 0] + X_ts[:, 1] * 0.5 + np.arange(n_samples) * 0.01 > 0).astype(int)

tscv = TimeSeriesSplit(n_splits=5)

print("=== 时间序列交叉验证划分示意 ===")
fig, ax = plt.subplots(figsize=(14, 4))

for i, (train_idx, test_idx) in enumerate(tscv.split(X_ts)):
    for idx in train_idx:
        ax.barh(i, 1, left=idx, color='#3498DB', alpha=0.6)
    for idx in test_idx:
        ax.barh(i, 1, left=idx, color='#E74C3C', alpha=0.8)

ax.set_xlabel('样本索引（时间顺序）', fontsize=12)
ax.set_ylabel('折数', fontsize=12)
ax.set_yticks(range(5))
ax.set_yticklabels([f'第{i+1}折' for i in range(5)])
ax.set_title('TimeSeriesSplit（训练数据始终在测试数据之前）', fontsize=14)
plt.tight_layout()
plt.show()

print("\n📌 时间序列CV的特点：")
print("  - 训练集始终在测试集之前（不会用未来数据训练）")
print("  - 训练集大小逐渐增加")
print("  - 避免了数据泄露问题")

---

## 📌 本节总结

| 方法 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **K折CV** | 数据分K份轮流验证 | 评估稳定，数据利用充分 | 需要训练K个模型 |
| **分层K折** | 保持每折类别比例 | 不平衡数据下更可靠 | 仅适用于分类 |
| **LOOCV** | K=N，每次留1个 | 训练数据最多，几乎无偏 | 计算量大，方差大 |
| **Grid Search** | 穷举所有参数组合 | 保证找到最优 | 组合爆炸 |
| **Random Search** | 随机采样参数组合 | 高效，覆盖面广 | 不保证最优 |

**关键要点：**
1. 交叉验证比单次划分更可靠
2. 分类任务优先使用StratifiedKFold
3. Pipeline + GridSearchCV是竞赛标准流程
4. 随机搜索在参数空间大时更高效
5. 时序数据必须使用TimeSeriesSplit防止数据泄露